# 03_Data_Cleaning

This notebook starts from `listing_clean_missing.parquet` and `sold_clean_missing.parquet`, which were created in a prior missing-value cleaning notebook. In that earlier step, high-missing non-core columns were removed. Therefore, this notebook focuses on duplicate-column removal, datatype conversion, invalid value checks, date consistency checks, and geographic data quality checks.

## Imports

In [37]:
import pandas as pd
import numpy as np

## Data Exploration

### Load the two datasets & Make working copies

In [38]:
sold = pd.read_parquet("data/processed/sold_clean_missing.parquet")
listing = pd.read_parquet("data/processed/listing_clean_missing.parquet")

In [39]:
sold_df = sold.copy()
listing_df = listing.copy()

In [40]:
sold_df.head()

,source_file,file_period,sort_date,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,...,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled
0,CRMLSSold202401.csv,202401,2024-01-01,OrangeCounty,OrangeCounty,"Carpet,Tile",False,False,854990.0,1040790377,...,True,2.0,Montebello Unified,90640,236.0,NaN,None,NaN,None,None
1,CRMLSSold202401.csv,202401,2024-01-01,NorthSanDiegoCounty,NorthSanDiegoCounty,"Laminate,Tile",False,False,399000.0,1044747152,...,False,2.0,Antelope Valley Union,93536,0.0,6882.0,None,NaN,None,None
2,CRMLSSold202401.csv,202401,2024-01-01,Southland,Southland,"Carpet,Tile",False,False,549999.0,1045213113,...,False,3.0,Palmdale,93552,0.0,7293.0,None,NaN,None,None
3,CRMLSSold202401.csv,202401,2024-01-01,SanDiego,SanDiego,None,True,False,1750000.0,1045523677,...,False,2.0,None,92054,695.0,13567.0,None,NaN,None,None
4,CRMLSSold202401.csv,202401,2024-01-01,PacificWest,PacificWest,"Laminate,Tile",False,False,535000.0,1046369026,...,False,2.0,Rialto Unified,92376,0.0,7700.0,None,NaN,None,None


In [41]:
listing_df.head()

,source_file,file_period,sort_date,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,...,LotSizeArea,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,BuyerOfficeName__dup2,AssociationFee,LotSizeSquareFeet,UnparsedAddress__dup2
0,CRMLSListing202401.csv,202401,2024-01-01,1399990.0,1043760653,joe@porchlightsocal.com,2024-03-15,1270000.0,Joseph,Vahedi,...,9897.0,2.0,None,2.0,Vista Unified,92083,eXp Realty of Southern CA,252.0,9897.0,123 Flores Lane
1,CRMLSListing202401.csv,202401,2024-01-01,789900.0,1053093534,kiva@kivakendrick.com,2024-02-26,800000.0,Kiva,Kendrick,...,5070.0,NaN,False,2.0,Los Angeles Unified,90746,R & R Realty,NaN,5070.0,19922 Enslow Drive
2,CRMLSListing202401.csv,202401,2024-01-01,1050000.0,1054014526,sarah.pearce@cbrealty.com,None,NaN,Sarah,Pearce,...,10019.0,NaN,False,NaN,None,92262,None,NaN,10019.0,2030 E Acacia Road
3,CRMLSListing202401.csv,202401,2024-01-01,1250000.0,1054022382,mricks@mypsrealtor.com,2024-02-26,1199000.0,Michael,Ricks,...,10454.0,NaN,False,2.0,None,92262,BHG Desert Lifestyle Properties,NaN,10454.0,1695 E Buena Vista Drive
4,CRMLSListing202401.csv,202401,2024-01-01,4750000.0,1054022420,james@larealproperty.com,None,NaN,James,Maxwell,...,3084.0,NaN,False,2.0,None,90291,None,NaN,3084.0,2915 S Grand Canal


### Check shape and columns

In [42]:
print("sold_df shape:", sold_df.shape)
print("listing_df shape:", listing_df.shape)

sold_df shape: (397603, 70)
listing_df shape: (540183, 74)


In [43]:
sold_cols = sold_df.columns.tolist()
listing_cols = listing_df.columns.tolist()

print("Sold columns:")
print(sold_cols)

print("\nListing columns:")
print(listing_cols)

Sold columns:
['source_file', 'file_period', 'sort_date', 'BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'AssociationFeeFrequency', 'ListingKeyNumeric', 'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate', 'StateOrProvince', 'MiddleOrJuniorSchool', 'FireplaceYN', 

## Data Cleaning

### Convert date fields to datetime

As an initial standardization step, the main transaction-related date fields are converted from raw string/object values into pandas datetime format. This transformation makes date values comparable across both datasets and prevents later timeline checks from failing because of inconsistent types.

Using `errors="coerce"` also turns malformed date strings into missing values, which is preferable to silently keeping invalid text in fields that will later be used for date-based quality checks.

In [44]:
date_cols = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]

for col in date_cols:
    if col in sold_df.columns:
        sold_df[col] = pd.to_datetime(sold_df[col], errors="coerce")
    if col in listing_df.columns:
        listing_df[col] = pd.to_datetime(listing_df[col], errors="coerce")

In [45]:
print("Sold Date Types:")
print(sold_df[[col for col in date_cols if col in sold_df.columns]].dtypes)
print("\nListing Date Types:")
print(listing_df[[col for col in date_cols if col in listing_df.columns]].dtypes)

Sold Date Types:
CloseDate                   datetime64[ns]
PurchaseContractDate        datetime64[ns]
ListingContractDate         datetime64[ns]
ContractStatusChangeDate    datetime64[ns]
dtype: object

Listing Date Types:
CloseDate                   datetime64[ns]
PurchaseContractDate        datetime64[ns]
ListingContractDate         datetime64[ns]
ContractStatusChangeDate    datetime64[ns]
dtype: object


### Remove redundant columns

#### Check duplicates within each dataset (duplicates and ending with `__dup` or `__dup2` columns)

In [46]:
# Check for duplicate columns
listing_dupes = listing_df.columns[listing_df.columns.duplicated()].tolist()
sold_dupes = sold_df.columns[sold_df.columns.duplicated()].tolist()

print("Listing duplicates:", listing_dupes)
print("Sold duplicates:", sold_dupes)

Listing duplicates: []
Sold duplicates: []


In [47]:
# Identify columns with __dup pattern
listing_dup_pattern = [col for col in listing_cols if '__dup' in col]
sold_dup_pattern = [col for col in sold_cols if '__dup' in col]

print("\nListing columns with __dup pattern:", listing_dup_pattern)
print("Sold columns with __dup pattern:", sold_dup_pattern)


Listing columns with __dup pattern: ['PropertyType__dup2', 'ListAgentFirstName__dup2', 'DaysOnMarket__dup2', 'LivingArea__dup2', 'Longitude__dup2', 'Latitude__dup2', 'ListPrice__dup2', 'ListAgentLastName__dup2', 'CloseDate__dup2', 'BuyerOfficeName__dup2', 'UnparsedAddress__dup2']
Sold columns with __dup pattern: []


#### Drop the duplicate columns based on the identified patterns

After identifying columns that were created as duplicate artifacts (for example, names ending in `__dup` or `__dup2`), those redundant versions are removed from the working DataFrames. This transformation keeps one canonical copy of each field so later analysis is not distorted by repeated information or ambiguous column selection.

In [48]:
# Drop the duplicate columns based on the identified patterns
sold_df = sold_df.drop(columns=sold_dup_pattern, errors="ignore")
listing_df = listing_df.drop(columns=listing_dup_pattern, errors="ignore")

print(f"Dropped {len(sold_dup_pattern)} columns from sold_df")
print(f"Dropped {len(listing_dup_pattern)} columns from listing_df")

Dropped 0 columns from sold_df
Dropped 11 columns from listing_df


In [49]:
# check the columns again after dropping duplicates
print("Sold columns after dropping duplicates:")
print(sold_df.columns.tolist())
print("\nListing columns after dropping duplicates:")
print(listing_df.columns.tolist())

Sold columns after dropping duplicates:
['source_file', 'file_period', 'sort_date', 'BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName', 'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName', 'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName', 'AssociationFeeFrequency', 'ListingKeyNumeric', 'MLSAreaMajor', 'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN', 'ParkingTotal', 'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric', 'ListingId', 'BathroomsTotalInteger', 'City', 'BedroomsTotal', 'ContractStatusChangeDate', 'PurchaseContractDate', 'ListingContractDate', 'StateOrProvince', 'MiddleOrJuni

Sold data has `latfilled` and `lonfilled`, check what thoes are and what they look like

In [50]:
print(sold_df["latfilled"].unique())
print(sold_df["lonfilled"].unique())

[None False True]
[None False True]


In [51]:
sold_df[["latfilled", "lonfilled"]].sample(10)

,latfilled,lonfilled
255773,None,None
32861,None,None
43804,None,None
98557,True,True
16663,None,None
106746,True,True
52589,None,None
336170,None,None
68139,True,True
110484,None,None


In [52]:
print(sold_df["latfilled"].value_counts(dropna=False))
print(sold_df["lonfilled"].value_counts(dropna=False))

latfilled
None     333719
True      33427
False     30457
Name: count, dtype: int64
lonfilled
None     333719
True      33427
False     30457
Name: count, dtype: int64


### Remove helper columns latfilled and lonfilled

The `latfilled` and `lonfilled` fields act as helper indicators showing whether coordinates may have been filled during earlier processing. They are not substantive property attributes, so they are removed from both datasets to keep the cleaned outputs focused on analytical variables rather than internal workflow metadata.


In [53]:
helper_cols_to_drop = ["latfilled", "lonfilled"]

sold_df = sold_df.drop(columns=helper_cols_to_drop, errors="ignore")
listing_df = listing_df.drop(columns=helper_cols_to_drop, errors="ignore")

In [54]:
print("Original Sold shape:", sold.shape, "Original Listing shape:", listing.shape)
print("Sold columns after dropping helper columns:", sold_df.shape, "Listing columns after dropping helper columns:", listing_df.shape)

Original Sold shape: (397603, 70) Original Listing shape: (540183, 74)
Sold columns after dropping helper columns: (397603, 68) Listing columns after dropping helper columns: (540183, 63)


### Standardize blank strings as missing values

The input files used in this notebook were already processed in a prior missing-value cleaning notebook, where high-missing non-core columns were removed. Here, the transformation is narrower: empty strings and whitespace-only text are converted to `NaN` so that missing values are represented consistently across the two datasets.

This matters because blank strings are not treated as missing by default, which can lead to misleading counts, inconsistent type conversion, and weaker quality checks later in the notebook.

In [55]:
# Standardize blank strings as missing values

for df in [sold_df, listing_df]:
    object_cols = df.select_dtypes(include="object").columns
    df[object_cols] = df[object_cols].replace(r"^\s*$", np.nan, regex=True)

#### Confirm date columns used later

Date columns were already converted to datetime earlier in the notebook. This section simply keeps the set of date fields explicit before the later timeline checks, so the notebook does not repeat the same transformation twice.

In [56]:
date_cols = [col for col in date_cols if col in sold_df.columns or col in listing_df.columns]
print("Date columns used in later checks:", date_cols)

Date columns used in later checks: ['CloseDate', 'PurchaseContractDate', 'ListingContractDate', 'ContractStatusChangeDate']


#### Ensure numeric fields are properly typed

Key numeric columns such as prices, coordinates, square footage, room counts, and lot-size measures are explicitly converted to numeric dtype. This transformation is necessary because values may arrive as strings after ingestion, and numeric validation rules only work reliably when these fields are stored as numbers.

Again, `errors="coerce"` is used so non-numeric entries become missing rather than remaining as invalid text.

In [57]:
# Check for numeric columns and convert them to numeric types
numeric_cols = [
    "file_period",
    "OriginalListPrice",
    "ListingKey",
    "ClosePrice",
    "Latitude",
    "Longitude",
    "LivingArea",
    "ListPrice",
    "DaysOnMarket",
    "ListingKeyNumeric",
    "ParkingTotal",
    "LotSizeAcres",
    "YearBuilt",
    "StreetNumberNumeric",
    "BathroomsTotalInteger",
    "BedroomsTotal",
    "Stories",
    "LotSizeArea",
    "MainLevelBedrooms",
    "GarageSpaces",
    "AssociationFee",
    "LotSizeSquareFeet",
    "BuyerAgencyCompensation"
]

for df in [sold_df, listing_df]:
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

Check numeric dtypes

In [58]:
print(sold_df[[col for col in numeric_cols if col in sold_df.columns]].dtypes)
print()
print(listing_df[[col for col in numeric_cols if col in listing_df.columns]].dtypes)

file_period                  int64
OriginalListPrice          float64
ListingKey                   int64
ClosePrice                 float64
Latitude                   float64
Longitude                  float64
LivingArea                 float64
ListPrice                  float64
DaysOnMarket                 int64
ListingKeyNumeric            int64
ParkingTotal               float64
LotSizeAcres               float64
YearBuilt                  float64
StreetNumberNumeric        float64
BathroomsTotalInteger      float64
BedroomsTotal              float64
Stories                    float64
LotSizeArea                float64
MainLevelBedrooms          float64
GarageSpaces               float64
AssociationFee             float64
LotSizeSquareFeet          float64
BuyerAgencyCompensation    float64
dtype: object

file_period                  int64
OriginalListPrice          float64
ListingKey                   int64
ClosePrice                 float64
Latitude                   float64
Longi

Missing value summary

In [59]:
print("Sold missing values:")
print(sold_df.isna().sum().sort_values(ascending=False).head(20))

print("\nListing missing values:")
print(listing_df.isna().sum().sort_values(ascending=False).head(20))

Sold missing values:
BuyerAgencyCompensation        351478
BuyerAgencyCompensationType    351467
ElementarySchool               344627
MiddleOrJuniorSchool           344273
HighSchool                     328353
CoListAgentFirstName           306611
CoListAgentLastName            306376
CoListOfficeName               303521
SubdivisionName                249875
AssociationFeeFrequency        232049
MainLevelBedrooms              166841
Flooring                       142590
HighSchoolDistrict             108857
AssociationFee                  90998
Stories                         61503
AttachedGarageYN                60679
MLSAreaMajor                    53122
BuyerAgentAOR                   49091
ListAgentAOR                    46186
Levels                          38569
dtype: int64

Listing missing values:
ElementarySchool               475845
MiddleOrJuniorSchool           475701
BuyerAgencyCompensation        461190
BuyerAgencyCompensationType    461170
HighSchool                   

### *Flag* invalid numeric values

This step creates boolean flag columns for clearly implausible numeric values, such as nonpositive close price or living area and negative counts for market days, bedrooms, or bathrooms. The goal is to document which rows violate basic domain rules before any records are removed.

A combined `any_invalid_numeric_flag` is also created so the notebook can later filter records in a transparent and reproducible way.

In [60]:
for df in [sold_df, listing_df]:
    closeprice = df.get("ClosePrice", pd.Series(np.nan, index=df.index))
    livingarea = df.get("LivingArea", pd.Series(np.nan, index=df.index))
    daysonmarket = df.get("DaysOnMarket", pd.Series(np.nan, index=df.index))
    bedroomstotal = df.get("BedroomsTotal", pd.Series(np.nan, index=df.index))
    bathroomstotal = df.get("BathroomsTotalInteger", pd.Series(np.nan, index=df.index))

    df["invalid_closeprice_flag"] = closeprice.notna() & (closeprice <= 0)
    df["invalid_livingarea_flag"] = livingarea.notna() & (livingarea <= 0)
    df["invalid_daysonmarket_flag"] = daysonmarket.notna() & (daysonmarket < 0)
    df["invalid_bedrooms_flag"] = bedroomstotal.notna() & (bedroomstotal < 0)
    df["invalid_bathrooms_flag"] = bathroomstotal.notna() & (bathroomstotal < 0)

    df["any_invalid_numeric_flag"] = (
        df["invalid_closeprice_flag"]
        | df["invalid_livingarea_flag"]
        | df["invalid_daysonmarket_flag"]
        | df["invalid_bedrooms_flag"]
        | df["invalid_bathrooms_flag"]
    )

In [61]:
num_invalid_sold = sold_df["any_invalid_numeric_flag"].sum()
num_invalid_listing = listing_df["any_invalid_numeric_flag"].sum()
print(f"Sold dataset has {num_invalid_sold} of invalid numeric values.")
print(f"Listing dataset has {num_invalid_listing} of invalid numeric values.")

Sold dataset has 191 of invalid numeric values.
Listing dataset has 388 of invalid numeric values.


### *Check* invalid numeric counts

In [62]:
invalid_cols = [
    "invalid_closeprice_flag",
    "invalid_livingarea_flag",
    "invalid_daysonmarket_flag",
    "invalid_bedrooms_flag",
    "invalid_bathrooms_flag",
    "any_invalid_numeric_flag"
]

print("Sold invalid numeric counts:")
print(sold_df[invalid_cols].sum())

print("\nListing invalid numeric counts:")
print(listing_df[invalid_cols].sum())

Sold invalid numeric counts:
invalid_closeprice_flag        1
invalid_livingarea_flag      144
invalid_daysonmarket_flag     46
invalid_bedrooms_flag          0
invalid_bathrooms_flag         0
any_invalid_numeric_flag     191
dtype: int64

Listing invalid numeric counts:
invalid_closeprice_flag        0
invalid_livingarea_flag      359
invalid_daysonmarket_flag     29
invalid_bedrooms_flag          0
invalid_bathrooms_flag         0
any_invalid_numeric_flag     388
dtype: int64


In [63]:
cols_to_look = [
    "ClosePrice", "LivingArea", "DaysOnMarket", "BedroomsTotal", "BathroomsTotalInteger", 
    "any_invalid_numeric_flag"
]

print("Sold sample:")
print(sold_df.loc[sold_df["any_invalid_numeric_flag"], cols_to_look].head(10))
print("\nListing sample:")
print(listing_df.loc[listing_df["any_invalid_numeric_flag"], cols_to_look].head(10))

Sold sample:
       ClosePrice  LivingArea  DaysOnMarket  BedroomsTotal  \
7209    8000000.0         0.0            62            4.0   
7518    1950000.0      2100.0           -36            3.0   
12590    700000.0      1382.0           -10            4.0   
14483   9000000.0         0.0            76            4.0   
20260   1850000.0         0.0            94            4.0   
20261   1930000.0         0.0           128            4.0   
21056    372000.0      1488.0           -10            3.0   
21818    899000.0      2533.0           -13            2.0   
23320    476985.0      1342.0           -13            3.0   
24545    715000.0      2284.0           -10            4.0   

       BathroomsTotalInteger  any_invalid_numeric_flag  
7209                     5.0                      True  
7518                     0.0                      True  
12590                    2.0                      True  
14483                    3.0                      True  
20260              

### Date consistency checks

These transformations add flag columns that test whether the ordering of major transaction dates makes sense. Records are marked when listing dates occur after close dates, when purchase contract dates occur after close dates, or when the purchase contract date appears before the listing contract date.

The purpose is to preserve an auditable record of temporal inconsistencies instead of dropping suspicious rows without showing why they were considered invalid.

**Required flags:**
- listing_after_close_flag
- purchase_after_close_flag
- negative_timeline_flag
- any_date_issue_flag

In [64]:
for df in [sold_df, listing_df]:
    listing_date = df.get("ListingContractDate", pd.Series(np.nan, index=df.index))
    purchase_date = df.get("PurchaseContractDate", pd.Series(np.nan, index=df.index))
    close_date = df.get("CloseDate", pd.Series(np.nan, index=df.index))

    df["listing_after_close_flag"] = (
        listing_date.notna() &
        close_date.notna() &
        (listing_date > close_date)
    )

    df["purchase_after_close_flag"] = (
        purchase_date.notna() &
        close_date.notna() &
        (purchase_date > close_date)
    )

    df["negative_timeline_flag"] = (
        listing_date.notna() &
        purchase_date.notna() &
        (purchase_date < listing_date)
    )

    df["any_date_issue_flag"] = (
        df["listing_after_close_flag"]
        | df["purchase_after_close_flag"]
        | df["negative_timeline_flag"]
    )

#### Count date issues

In [65]:
date_flag_cols = [
    "listing_after_close_flag",
    "purchase_after_close_flag",
    "negative_timeline_flag",
    "any_date_issue_flag"
]

print("Sold date issue counts:")
print(sold_df[date_flag_cols].sum())

print("\nListing date issue counts:")
print(listing_df[date_flag_cols].sum())

Sold date issue counts:
listing_after_close_flag      58
purchase_after_close_flag    240
negative_timeline_flag       261
any_date_issue_flag          501
dtype: int64

Listing date issue counts:
listing_after_close_flag      72
purchase_after_close_flag    265
negative_timeline_flag       271
any_date_issue_flag          535
dtype: int64


### Geographic data checks

This section creates geographic quality flags for missing coordinates, zero coordinates, positive longitude values, and latitude/longitude pairs that fall outside a plausible California range. These checks are used to identify records whose spatial information is incomplete or unrealistic for the study area.

Creating explicit flags first makes the later filtering step easier to justify and summarize.

In [66]:
for df in [sold_df, listing_df]:
    latitude = df.get("Latitude", pd.Series(np.nan, index=df.index))
    longitude = df.get("Longitude", pd.Series(np.nan, index=df.index))

    df["missing_coordinates_flag"] = latitude.isna() | longitude.isna()

    df["zero_coordinates_flag"] = ((latitude == 0) | (longitude == 0))

    df["positive_longitude_flag"] = longitude.notna() & (longitude > 0)

    df["out_of_state_or_implausible_flag"] = (
        latitude.notna() & longitude.notna() &
        (
            (latitude < 32) | (latitude > 42.5) |
            (longitude < -125) | (longitude > -114)
        )
    )

    df["any_geo_issue_flag"] = (
        df["missing_coordinates_flag"]
        | df["zero_coordinates_flag"]
        | df["positive_longitude_flag"]
        | df["out_of_state_or_implausible_flag"]
    )

#### Count geographic issues

In [67]:
geo_flag_cols = [
    "missing_coordinates_flag",
    "zero_coordinates_flag",
    "positive_longitude_flag",
    "out_of_state_or_implausible_flag",
    "any_geo_issue_flag"
]

print("Sold geographic issue counts:")
print(sold_df[geo_flag_cols].sum())

print("\nListing geographic issue counts:")
print(listing_df[geo_flag_cols].sum())

Sold geographic issue counts:
missing_coordinates_flag            15822
zero_coordinates_flag                  25
positive_longitude_flag                29
out_of_state_or_implausible_flag       84
any_geo_issue_flag                  15906
dtype: int64

Listing geographic issue counts:
missing_coordinates_flag            80145
zero_coordinates_flag                  60
positive_longitude_flag                85
out_of_state_or_implausible_flag      287
any_geo_issue_flag                  80432
dtype: int64


View sample flagged records

In [68]:
# Display rows for sold data
sold_df.loc[
    sold_df["any_invalid_numeric_flag"]
    | sold_df["any_date_issue_flag"]
    | sold_df["any_geo_issue_flag"]
].head(10)

,source_file,file_period,sort_date,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,...,any_invalid_numeric_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,any_date_issue_flag,missing_coordinates_flag,zero_coordinates_flag,positive_longitude_flag,out_of_state_or_implausible_flag,any_geo_issue_flag
3,CRMLSSold202401.csv,202401,2024-01-01,SanDiego,SanDiego,None,True,False,1750000.0,1045523677,...,False,False,False,False,False,True,False,False,False,True
7,CRMLSSold202401.csv,202401,2024-01-01,SanDiego,SanDiego,Wood,True,False,184500.0,1048590625,...,False,False,False,False,False,True,False,False,False,True
9,CRMLSSold202401.csv,202401,2024-01-01,SanDiego,SanDiego,None,True,False,1600000.0,1054052014,...,False,True,False,True,True,True,False,False,False,True
11,CRMLSSold202401.csv,202401,2024-01-02,Mlslistings,Mlslistings,None,False,None,8500000.0,1033955925,...,False,False,False,False,False,True,False,False,False,True
23,CRMLSSold202401.csv,202401,2024-01-02,Mlslistings,Mlslistings,"Carpet,Tile",True,False,469000.0,1038308808,...,False,False,False,False,False,True,False,False,False,True
27,CRMLSSold202401.csv,202401,2024-01-02,BayEast,BayEast,"Laminate,SeeRemarks",None,False,435000.0,1038588170,...,False,False,False,False,False,True,False,False,False,True
29,CRMLSSold202401.csv,202401,2024-01-02,Mlslistings,Mlslistings,None,True,None,925000.0,1038735217,...,False,False,False,False,False,True,False,False,False,True
42,CRMLSSold202401.csv,202401,2024-01-02,Mlslistings,Mlslistings,None,False,None,759000.0,1040987531,...,False,False,False,False,False,True,False,False,False,True
51,CRMLSSold202401.csv,202401,2024-01-02,Mlslistings,Mlslistings,"Carpet,Tile",True,False,1050000.0,1041333510,...,False,False,False,False,False,True,False,False,False,True
52,CRMLSSold202401.csv,202401,2024-01-02,Mlslistings,Mlslistings,None,False,None,969800.0,1041402771,...,False,False,False,False,False,True,False,False,False,True


In [69]:
# Display rows for listing data
listing_df.loc[
    listing_df["any_invalid_numeric_flag"]
    | listing_df["any_date_issue_flag"]
    | listing_df["any_geo_issue_flag"]
].head(10)

,source_file,file_period,sort_date,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,ListAgentFirstName,ListAgentLastName,...,any_invalid_numeric_flag,listing_after_close_flag,purchase_after_close_flag,negative_timeline_flag,any_date_issue_flag,missing_coordinates_flag,zero_coordinates_flag,positive_longitude_flag,out_of_state_or_implausible_flag,any_geo_issue_flag
5,CRMLSListing202401.csv,202401,2024-01-01,720000.0,1054022439,jean@centermacsd.com,2024-02-21,740000.0,Jean,Riley,...,False,False,False,False,False,True,False,False,False,True
15,CRMLSListing202401.csv,202401,2024-01-01,1524900.0,1054022640,dimeisimswpr@yahoo.com,2024-02-15,1530000.0,DiMei,Sims,...,False,False,False,False,False,True,False,False,False,True
16,CRMLSListing202401.csv,202401,2024-01-01,649000.0,1054022809,jomeara@bhhscal.com,2024-01-29,650000.0,Janell,OMeara,...,False,False,False,False,False,True,False,False,False,True
18,CRMLSListing202401.csv,202401,2024-01-01,15888000.0,1054022833,stilla.raissi@cbnorcal.com,NaT,NaN,Estila,Raissi,...,False,False,False,False,False,True,False,False,False,True
19,CRMLSListing202401.csv,202401,2024-01-01,1149900.0,1054022870,valerie@valerieann.net,2024-05-06,1020000.0,Valerie,Smith,...,False,False,False,False,False,True,False,False,False,True
22,CRMLSListing202401.csv,202401,2024-01-01,2499000.0,1054023129,cheri.mosdell@gmail.com,2024-02-13,2335000.0,Cheri,Mosdell,...,False,False,False,False,False,True,False,False,False,True
23,CRMLSListing202401.csv,202401,2024-01-01,795000.0,1054023218,mike_cascio@sbcglobal.net,2024-01-30,900000.0,Mike,Cascio,...,False,False,False,False,False,True,False,False,False,True
26,CRMLSListing202401.csv,202401,2024-01-01,620000.0,1054023259,janetemass@gmail.com,2024-01-25,630000.0,Janet,Mass,...,False,False,False,False,False,True,False,False,False,True
28,CRMLSListing202401.csv,202401,2024-01-01,860000.0,1054023399,brendan@the5starteam.com,2024-02-26,853000.0,Brendan,Moran,...,False,False,False,False,False,True,False,False,False,True
30,CRMLSListing202401.csv,202401,2024-01-01,2495000.0,1054023558,christy@crestmontrealty.com,2024-04-05,2250000.0,Christy,Ward,...,False,False,False,False,False,True,False,False,False,True


### Final Filter

Once all numeric, date, and geographic issue flags have been created, the notebook removes rows with any flagged problem to produce the final cleaned datasets. This transformation is the main record-level filtering step, and it is placed at the end so each removal reason has already been defined and counted.

In [70]:
# Remove rows with any of the identified issues for both datasets
sold_remove_mask = (
    sold_df["any_invalid_numeric_flag"]
    | sold_df["any_date_issue_flag"]
    | sold_df["any_geo_issue_flag"]
)

listing_remove_mask = (
    listing_df["any_invalid_numeric_flag"]
    | listing_df["any_date_issue_flag"]
    | listing_df["any_geo_issue_flag"]
)

sold_cleaned = sold_df.loc[
    ~sold_remove_mask
].copy()

listing_cleaned = listing_df.loc[
    ~listing_remove_mask
].copy()

### Row count summary

In [71]:
print("Sold rows flagged by issue type:")
print(pd.Series({
    "invalid_numeric": sold_df["any_invalid_numeric_flag"].sum(),
    "date_issue": sold_df["any_date_issue_flag"].sum(),
    "geo_issue": sold_df["any_geo_issue_flag"].sum(),
    "rows_removed_total": sold_remove_mask.sum()
} ))

print("Sold rows before cleaning:", len(sold_df))
print("Sold rows after cleaning:", len(sold_cleaned))
print("Sold rows removed:", len(sold_df) - len(sold_cleaned))

print("\nListing rows flagged by issue type:")
print(pd.Series({
    "invalid_numeric": listing_df["any_invalid_numeric_flag"].sum(),
    "date_issue": listing_df["any_date_issue_flag"].sum(),
    "geo_issue": listing_df["any_geo_issue_flag"].sum(),
    "rows_removed_total": listing_remove_mask.sum()
} ))

print("\nListing rows before cleaning:", len(listing_df))
print("Listing rows after cleaning:", len(listing_cleaned))
print("Listing rows removed:", len(listing_df) - len(listing_cleaned))

Sold rows flagged by issue type:
invalid_numeric         191
date_issue              501
geo_issue             15906
rows_removed_total    16512
dtype: int64
Sold rows before cleaning: 397603
Sold rows after cleaning: 381091
Sold rows removed: 16512

Listing rows flagged by issue type:
invalid_numeric         388
date_issue              535
geo_issue             80432
rows_removed_total    81167
dtype: int64

Listing rows before cleaning: 540183
Listing rows after cleaning: 459016
Listing rows removed: 81167


### Save outputs as csv. file

The final transformation result is written to `data/processed` as CSV files so the cleaned datasets can be reused in later notebooks or external tools. Saving the outputs here preserves a stable post-cleaning version of both tables for downstream analysis.

In [72]:
from pathlib import Path

output_dir = Path("data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

sold_cleaned.to_csv(output_dir / "sold_cleaned_final.csv", index=False)
listing_cleaned.to_csv(output_dir / "listing_cleaned_final.csv", index=False)

## Data Cleaning Summary

This notebook begins with datasets that were previously processed for high-missing columns. In this notebook, redundant duplicate columns were removed, blank strings were standardized as missing values, date columns were converted to datetime format, and key numeric columns were converted to numeric types.

Invalid numeric values were flagged for records with nonpositive close price or living area, negative days on market, and negative bedroom or bathroom counts. Date consistency flags were created to identify records where listing or purchase dates occurred after the close date, or where the purchase date occurred before the listing date. Geographic quality checks were also applied to flag missing coordinates, zero coordinates, positive longitude values, and implausible out-of-state coordinates.

After these checks, flagged invalid records were removed to create final cleaned, analysis-ready datasets for downstream analysis. The cleaned outputs were then saved as final CSV files.